In [1]:
!pip install rapidfuzz
!pip install wordcloud
!pip install seaborn

In [100]:
#Importación de librerías 

from rapidfuzz import process, fuzz
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from wordcloud import WordCloud
from pathlib import Path
from collections import Counter
import seaborn as sns
import re
import unicodedata
from functools import lru_cache


In [101]:
# se carga el dataset limpio previo al proceso de imputación
df_limpio = pd.read_csv("../data/processed/df_final.csv")
df_limpio

,file_name,n_legislatura,cuerpo,fecha,locutor,encabezado,intervencion,n_palabras,n_caracteres,año
0,47_2010-02-15_css.txt,47,css,2010-02-15,SEÑOR AMORÍN,DESCONOCIDO,"Sí, prometo. nor desempeñar debidamente el car...",20,119,2010
1,47_2010-02-15_css.txt,47,css,2010-02-15,SEÑORA PRESIDENTA,DESCONOCIDO,Queda usted investido del cargo de Senador. ¡B...,37,229,2010
2,47_2010-02-15_css.txt,47,css,2010-02-15,SEÑORA PRESIDENTA,DESCONOCIDO,Queda usted investido del cargo de Senador. ¡F...,34,222,2010
3,47_2010-02-15_css.txt,47,css,2010-02-15,SEÑORA PRESIDENTA,DESCONOCIDO,Queda usted investido del cargo de Senador. ¡B...,37,230,2010
4,47_2010-02-15_css.txt,47,css,2010-02-15,SEÑORA PRESIDENTA,DESCONOCIDO,Queda usted investido del cargo de Senador. ¡F...,34,225,2010
...,...,...,...,...,...,...,...,...,...,...
45161,50_2025-10-27_css.txt,50,css,2025-10-27,SEÑORA PRESIDENTA,9) RECTIFICACIÓN DE TRÁMITE,Queda usted investido del cargo de senador.\n–...,70,403,2025
45162,50_2025-10-27_css.txt,50,css,2025-10-27,SEÑORA PRESIDENTA,10) LUIS ALBERTO HEBER. SUBSIDIO,Léase la nota presentada por el exsenador Luis...,17,111,2025
45163,50_2025-10-27_css.txt,50,css,2025-10-27,SEÑORA PRESIDENTA,10) LUIS ALBERTO HEBER. SUBSIDIO,De conformidad con lo que establece el numeral...,58,322,2025
45164,50_2025-10-27_css.txt,50,css,2025-10-27,SEÑORA PRESIDENTA,11) SUSPENSIÓN DE LAS SESIONES ORDINARIAS,Léase una moción llegada a la mesa..\nSEÑOR SE...,41,245,2025


In [102]:
# contar locutores que tengan la palabra presid en el campo locutor
df_presid = df_limpio[df_limpio["locutor"].str.contains("president", case=False, na=False)]
df_presid

,file_name,n_legislatura,cuerpo,fecha,locutor,encabezado,intervencion,n_palabras,n_caracteres,año
1,47_2010-02-15_css.txt,47,css,2010-02-15,SEÑORA PRESIDENTA,DESCONOCIDO,Queda usted investido del cargo de Senador. ¡B...,37,229,2010
2,47_2010-02-15_css.txt,47,css,2010-02-15,SEÑORA PRESIDENTA,DESCONOCIDO,Queda usted investido del cargo de Senador. ¡F...,34,222,2010
3,47_2010-02-15_css.txt,47,css,2010-02-15,SEÑORA PRESIDENTA,DESCONOCIDO,Queda usted investido del cargo de Senador. ¡B...,37,230,2010
4,47_2010-02-15_css.txt,47,css,2010-02-15,SEÑORA PRESIDENTA,DESCONOCIDO,Queda usted investido del cargo de Senador. ¡F...,34,225,2010
5,47_2010-02-15_css.txt,47,css,2010-02-15,SEÑORA PRESIDENTA,DESCONOCIDO,Queda usted investido del cargo de Senador. ¡F...,34,227,2010
...,...,...,...,...,...,...,...,...,...,...
45161,50_2025-10-27_css.txt,50,css,2025-10-27,SEÑORA PRESIDENTA,9) RECTIFICACIÓN DE TRÁMITE,Queda usted investido del cargo de senador.\n–...,70,403,2025
45162,50_2025-10-27_css.txt,50,css,2025-10-27,SEÑORA PRESIDENTA,10) LUIS ALBERTO HEBER. SUBSIDIO,Léase la nota presentada por el exsenador Luis...,17,111,2025
45163,50_2025-10-27_css.txt,50,css,2025-10-27,SEÑORA PRESIDENTA,10) LUIS ALBERTO HEBER. SUBSIDIO,De conformidad con lo que establece el numeral...,58,322,2025
45164,50_2025-10-27_css.txt,50,css,2025-10-27,SEÑORA PRESIDENTA,11) SUSPENSIÓN DE LAS SESIONES ORDINARIAS,Léase una moción llegada a la mesa..\nSEÑOR SE...,41,245,2025


In [103]:
#Se quitan de df_limpio las filas donde el locutor contenga la palabra "president"
df_limpio = df_limpio[~df_limpio["locutor"].str.contains("president", case=False, na=False)]

In [104]:
df_parlamentarios = pd.read_csv("../data/raw/parlamentarios/ag_actuantes_legislaturas_sexo_imputado.csv")

In [105]:
# Se filtran los parlamentarios pertenecientes a la camara CSS
df_parlamentarios_css = df_parlamentarios[df_parlamentarios['camara'] == 'CSS']
df_parlamentarios_css

,fecha,camara,partido,apellido_norm,nombre_norm,sexo,sustituye_a
0,2010-02-15,CSS,FRENTE AMPLIO,topolansky,lucia,F,NaN
1,2010-02-15,CSS,NACIONAL,abreu,sergio,M,NaN
2,2010-02-15,CSS,FRENTE AMPLIO,agazzi,ernesto,M,NaN
3,2010-02-15,CSS,COLORADO,amorin,jose,M,NaN
4,2010-02-15,CSS,FRENTE AMPLIO,baraibar,carlos,M,NaN
...,...,...,...,...,...,...,...
710791,2025-11-22,CSS,COLORADO,zubia,gustavo,M,NaN
710800,2025-11-22,CSS,FRENTE AMPLIO,badin vidal,cecilia,F,NaN
710823,2025-11-22,CSS,FRENTE AMPLIO,garlo alonsoperez,joaquin,M,NaN
710829,2025-11-22,CSS,FRENTE AMPLIO,guerrero,gustavo,M,NaN


In [106]:
if "n_legislatura" not in df_parlamentarios_css.columns:
    # 1) Convertí una sola vez a datetime (si ya es datetime)
    fecha = pd.to_datetime(df_parlamentarios_css["fecha"], errors="coerce")

    # 2) Bins: inicio de cada legislatura + el final (inicio de la “siguiente”)
    bins = pd.to_datetime([
        "2010-02-15",
        "2015-02-15",
        "2020-02-15",
        "2025-02-15",
        "2030-02-15",
    ])

    labels = pd.array([47, 48, 49, 50], dtype="Int64")

    df_parlamentarios_css["n_legislatura"] = (
        pd.cut(fecha, bins=bins, right=False, labels=labels)
        .astype("Int64")
    )


C:\Users\flala\AppData\Local\Temp\ipykernel_2880\3801259117.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_parlamentarios_css["n_legislatura"] = (


In [107]:
ambiguedad = (
    df_parlamentarios_css
    .groupby("apellido_norm")
    .agg(
        n_partidos=("partido", "nunique")
    )
    .reset_index()
)

apellidos_conflictivos = ambiguedad[
    ambiguedad["n_partidos"] > 1
]["apellido_norm"]



In [108]:
df_homonimos = df_parlamentarios_css[
    df_parlamentarios_css["apellido_norm"].isin(apellidos_conflictivos)
].sort_values(["apellido_norm", "fecha"])


df_homonimos_unicos = (
    df_homonimos
    .drop_duplicates(
        subset=["partido", "apellido_norm", "nombre_norm", "sexo"]
    )
    .sort_values(["apellido_norm", "nombre_norm"])
)


In [109]:
df_homonimos_unicos

,fecha,camara,partido,apellido_norm,nombre_norm,sexo,sustituye_a,n_legislatura
377674,2018-10-23,CSS,NACIONAL,alvarez lopez,maria,F,"la Senadora Alonso, Verónica durante la licenc...",48
2878,2010-03-10,CSS,FRENTE AMPLIO,alvarez lopez,pablo,M,NaN,47
211744,2015-04-15,CSS,COLORADO,bianchi,daniel,M,"la Senadora Montaner, Martha durante la licenc...",48
6361,2010-04-06,CSS,FRENTE AMPLIO,bianchi,eleonora,F,NaN,47
439645,2020-02-15,CSS,NACIONAL,bianchi,graciela,F,NaN,49
10894,2010-05-11,CSS,COLORADO,cardoso,german,M,NaN,47
66349,2011-07-13,CSS,NACIONAL,cardoso,jose,M,NaN,47
635525,2024-04-17,CSS,NACIONAL,correa,julio,M,NaN,49
11205,2010-05-13,CSS,COLORADO,correa,marco,M,NaN,47
444487,2020-03-24,CSS,NACIONAL,garcia,alem,M,NaN,49


In [110]:
apellidos_ambiguos = set(df_homonimos_unicos["apellido_norm"].unique())
apellidos_ambiguos

{'alvarez lopez',
 'bianchi',
 'cardoso',
 'correa',
 'garcia',
 'gonzalez',
 'moreira',
 'pereira',
 'pereyra',
 'rodriguez',
 'sanguinetti',
 'saravia',
 'silva',
 'vega'}

In [111]:
# Se construye un diccionario de DataFrames agrupados por sexo.
# Cada clave corresponde a un sexo y su valor contiene los parlamentarios asociados.
# Para evitar sesgos por repetición de registros históricos, se eliminan duplicados.
# La deduplicación se realiza por la combinación apellido–partido.
# Esto asegura que cada apellido contribuya una sola vez al proceso de matching.

parl_dict_sexo = {
    sexo: df.drop_duplicates(subset=["apellido_norm", "partido"])
    for sexo, df in df_parlamentarios_css.groupby("sexo")
}

In [112]:
def norm_text(s: str) -> str:
    s = str(s).strip()
    s = unicodedata.normalize("NFKD", s)
    s = "".join(c for c in s if not unicodedata.combining(c))
    s = s.lower()
    s = re.sub(r"[^a-z\s]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s


In [113]:
TITULOS = r"(SENOR|SENORA|PRESIDENTE|VICEPRESIDENTE|SECRETARIO)"

def extraer_apellido_locutor(locutor):
    if pd.isna(locutor):
        return pd.NA
    s = norm_text(locutor)
    s = re.sub(rf"^{TITULOS}\s+", "", s, flags=re.IGNORECASE).strip()
    return s if s else pd.NA


In [114]:
df_limpio["fecha"] = pd.to_datetime(df_limpio["fecha"])
df_limpio.columns = df_limpio.columns.str.strip()

df_limpio["apellido_locutor_norm"] = df_limpio["locutor"].apply(extraer_apellido_locutor)



C:\Users\flala\AppData\Local\Temp\ipykernel_2880\2880585394.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_limpio["fecha"] = pd.to_datetime(df_limpio["fecha"])
C:\Users\flala\AppData\Local\Temp\ipykernel_2880\2880585394.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_limpio["apellido_locutor_norm"] = df_limpio["locutor"].apply(extraer_apellido_locutor)


In [115]:
PARTICULAS = {"de","del","la","las","los","y","da","do","dos","das","van","von"}

def apellido_key(apellido_norm):
    if pd.isna(apellido_norm):
        return pd.NA
    toks = norm_text(apellido_norm).split()
    for t in toks:
        if t not in PARTICULAS:
            return t
    return toks[0] if toks else pd.NA


In [116]:
df_limpio["apellido_key"] = df_limpio["apellido_locutor_norm"].apply(apellido_key)

df_parlamentarios_css["fecha"] = pd.to_datetime(df_parlamentarios_css["fecha"])
df_parlamentarios_css.columns = df_parlamentarios_css.columns.str.strip()

# IMPORTANTE: asegurá que apellido_norm en parlamentarios esté normalizado igual (si no, normalizalo)
df_parlamentarios_css["apellido_norm"] = df_parlamentarios_css["apellido_norm"].apply(norm_text)

df_parlamentarios_css["apellido_key"] = df_parlamentarios_css["apellido_norm"].apply(apellido_key)


C:\Users\flala\AppData\Local\Temp\ipykernel_2880\2828860399.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_limpio["apellido_key"] = df_limpio["apellido_locutor_norm"].apply(apellido_key)
C:\Users\flala\AppData\Local\Temp\ipykernel_2880\2828860399.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_parlamentarios_css["fecha"] = pd.to_datetime(df_parlamentarios_css["fecha"])
C:\Users\flala\AppData\Local\Temp\ipykernel_2880\2828860399.py:7: SettingWithCopyWarning: 
A value is trying to be set on a c

In [117]:
if "n_legislatura" not in df_parlamentarios_css.columns:
    # 1) Convertí una sola vez a datetime (si ya es datetime)
    fecha = pd.to_datetime(df_parlamentarios_css["fecha"], errors="coerce")

    # 2) Bins: inicio de cada legislatura + el final (inicio de la “siguiente”)
    bins = pd.to_datetime([
        "2010-02-15",
        "2015-02-15",
        "2020-02-15",
        "2025-02-15",
        "2030-02-15",
    ])

    labels = pd.array([47, 48, 49, 50], dtype="Int64")

    df_parlamentarios_css["n_legislatura"] = (
        pd.cut(fecha, bins=bins, right=False, labels=labels)
        .astype("Int64")
    )


In [118]:
LEGISLATURES = [
    {"value": 50, "desde": "2025-02-15", "hasta": "2030-02-14"},
    {"value": 49, "desde": "2020-02-15", "hasta": "2025-02-14"},
    {"value": 48, "desde": "2015-02-15", "hasta": "2020-02-14"},
    {"value": 47, "desde": "2010-02-15", "hasta": "2015-02-14"},
]

def asignar_legislatura(fecha):
    if pd.isna(fecha):
        return pd.NA
    for leg in LEGISLATURES:
        if pd.to_datetime(leg["desde"]) <= fecha <= pd.to_datetime(leg["hasta"]):
            return leg["value"]
    return pd.NA

if "n_legislatura" not in df_parlamentarios_css.columns:
    df_parlamentarios_css["n_legislatura"] = df_parlamentarios_css["fecha"].apply(asignar_legislatura).astype("Int64")


In [119]:
amb_full = (
    df_parlamentarios_css.groupby("apellido_norm")["partido"]
    .nunique()
    .reset_index(name="n_partidos")
)
apellidos_conflictivos_full = set(amb_full.loc[amb_full["n_partidos"] > 1, "apellido_norm"])


In [120]:
amb_key = (
    df_parlamentarios_css.groupby("apellido_key")["partido"]
    .nunique()
    .reset_index(name="n_partidos")
)
apellidos_conflictivos_key = set(amb_key.loc[amb_key["n_partidos"] > 1, "apellido_key"])


In [121]:
for col in ["partido_imputado", "metodo_imputacion", "score_match"]:
    if col not in df_limpio.columns:
        df_limpio[col] = pd.NA


C:\Users\flala\AppData\Local\Temp\ipykernel_2880\1380075392.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_limpio[col] = pd.NA
C:\Users\flala\AppData\Local\Temp\ipykernel_2880\1380075392.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_limpio[col] = pd.NA
C:\Users\flala\AppData\Local\Temp\ipykernel_2880\1380075392.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the docu

In [122]:
# 1) Aseguramos que locutor sea string y limpiamos espacios
s = df_limpio["locutor"].fillna("").astype(str).str.strip()

# 2) Normalizamos a MAYÚSCULAS (y por las dudas, también cubrimos "SENOR/SENORA" sin ñ)
s_up = s.str.upper()

df_limpio["sexo_locutor"] = np.select(
    [
        s_up.str.startswith("SEÑORA") | s_up.str.startswith("SENORA"),
        s_up.str.startswith("SEÑOR")  | s_up.str.startswith("SENOR"),
    ],
    ["F", "M"],
    default=pd.NA
)


C:\Users\flala\AppData\Local\Temp\ipykernel_2880\2718703330.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_limpio["sexo_locutor"] = np.select(


In [123]:
parl_exacto = df_parlamentarios_css[[
    "apellido_norm", "sexo", "fecha", "n_legislatura", "partido"
]].drop_duplicates()

df_limpio = df_limpio.merge(
    parl_exacto,
    left_on=["apellido_locutor_norm", "sexo_locutor", "fecha", "n_legislatura"],
    right_on=["apellido_norm", "sexo", "fecha", "n_legislatura"],
    how="left",
    suffixes=("", "_parl")
)

mask = df_limpio["partido"].notna()
df_limpio.loc[mask, "partido_imputado"] = df_limpio.loc[mask, "partido"]
df_limpio.loc[mask, "metodo_imputacion"] = "exacto"
df_limpio.loc[mask, "score_match"] = 100


In [124]:
parl_leg_unico_full = (
    df_parlamentarios_css
    .groupby(["apellido_norm", "sexo", "n_legislatura"])["partido"]
    .nunique()
    .reset_index(name="n_partidos")
)
parl_leg_unico_full = parl_leg_unico_full[parl_leg_unico_full["n_partidos"] == 1].drop(columns="n_partidos")

parl_leg_unico_full = parl_leg_unico_full.merge(
    df_parlamentarios_css[["apellido_norm","sexo","n_legislatura","partido"]].drop_duplicates(),
    on=["apellido_norm","sexo","n_legislatura"],
    how="left"
)

mask_pend = (
    df_limpio["partido_imputado"].isna() &
    (~df_limpio["apellido_locutor_norm"].isin(apellidos_conflictivos_full))
)

df_limpio = df_limpio.merge(
    parl_leg_unico_full,
    left_on=["apellido_locutor_norm", "sexo_locutor", "n_legislatura"],
    right_on=["apellido_norm", "sexo", "n_legislatura"],
    how="left",
    suffixes=("", "_legfull")
)

mask2 = mask_pend & df_limpio["partido_legfull"].notna()
df_limpio.loc[mask2, "partido_imputado"] = df_limpio.loc[mask2, "partido_legfull"]
df_limpio.loc[mask2, "metodo_imputacion"] = "legislatura_unica_full"
df_limpio.loc[mask2, "score_match"] = 95


In [125]:
parl_leg_unico_key = (
    df_parlamentarios_css
    .groupby(["apellido_key", "sexo", "n_legislatura"])["partido"]
    .nunique()
    .reset_index(name="n_partidos")
)
parl_leg_unico_key = parl_leg_unico_key[parl_leg_unico_key["n_partidos"] == 1].drop(columns="n_partidos")

parl_leg_unico_key = parl_leg_unico_key.merge(
    df_parlamentarios_css[["apellido_key","sexo","n_legislatura","partido"]].drop_duplicates(),
    on=["apellido_key","sexo","n_legislatura"],
    how="left"
)

mask_pend = (
    df_limpio["partido_imputado"].isna() &
    (~df_limpio["apellido_key"].isin(apellidos_conflictivos_key))
)

df_limpio = df_limpio.merge(
    parl_leg_unico_key,
    left_on=["apellido_key", "sexo_locutor", "n_legislatura"],
    right_on=["apellido_key", "sexo", "n_legislatura"],
    how="left",
    suffixes=("", "_legkey")
)

mask3 = mask_pend & df_limpio["partido_legkey"].notna()
df_limpio.loc[mask3, "partido_imputado"] = df_limpio.loc[mask3, "partido_legkey"]
df_limpio.loc[mask3, "metodo_imputacion"] = "legislatura_unica_key"
df_limpio.loc[mask3, "score_match"] = 90


In [126]:
# Pendientes reales para fuzzy
df_pend = df_limpio[df_limpio["partido_imputado"].isna()].copy()

# candidatos por (sexo, legislatura), usando apellido_norm completo
cand = (
    df_parlamentarios_css
    .drop_duplicates(subset=["apellido_norm", "partido", "sexo", "n_legislatura"])
)
candidatos_por_grupo = {
    (s, l): g for (s, l), g in cand.groupby(["sexo", "n_legislatura"])
}

@lru_cache(maxsize=50000)
def fuzzy_best(apellido, sexo, leg):
    g = candidatos_por_grupo.get((sexo, leg))
    if g is None or g.empty:
        return None
    return process.extractOne(apellido, g["apellido_norm"], scorer=fuzz.token_sort_ratio)

def aplicar_fuzzy_row(r, score_min=85):
    # si no hay apellido completo, probar con key como fallback
    apellido = r["apellido_locutor_norm"]
    sexo = r["sexo_locutor"]
    leg = r["n_legislatura"]
    if pd.isna(apellido) or pd.isna(sexo) or pd.isna(leg):
        return pd.Series([pd.NA, pd.NA])

    m = fuzzy_best(apellido, sexo, int(leg))
    if m and m[1] >= score_min:
        g = candidatos_por_grupo[(sexo, int(leg))]
        filas = g[g["apellido_norm"] == m[0]]
        partidos = filas["partido"].unique()
        if len(partidos) == 1:
            return pd.Series([partidos[0], m[1]])

    return pd.Series([pd.NA, pd.NA])

# aplicar fuzzy SOLO a pendientes
fuzzy_out = df_pend.apply(aplicar_fuzzy_row, axis=1)
df_limpio.loc[df_pend.index, ["partido_imputado", "score_match"]] = fuzzy_out.values
df_limpio.loc[df_pend.index[fuzzy_out[0].notna()], "metodo_imputacion"] = "fuzzy"


In [127]:
columnas_finales = [
    "file_name", "n_legislatura", "cuerpo", "fecha", "locutor", "encabezado",
    "intervencion", "n_palabras", "n_caracteres", "año",
    "sexo_locutor", "apellido_locutor_norm", "apellido_key",
    "partido_imputado", "metodo_imputacion", "score_match"
]
df_limpio = df_limpio[[c for c in columnas_finales if c in df_limpio.columns]]


In [128]:
# Asegurar datetime
df_limpio["fecha"] = pd.to_datetime(df_limpio["fecha"])
df_parlamentarios_css["fecha"] = pd.to_datetime(df_parlamentarios_css["fecha"])

# (Opcional) si tenés cámara/cuerpo en ambos, mejor: agrega "camara" al groupby
# Ej: en df_limpio 'cuerpo' y en parl 'camara' -> normalizalos a una sola columna
# df_limpio["camara"] = df_limpio["cuerpo"].str.upper()
# df_parlamentarios_css["camara"] = df_parlamentarios_css["camara"].str.upper()

actuo_ese_dia = (
    df_parlamentarios_css
    .groupby(["fecha", "apellido_key", "sexo"])["partido"]
    .nunique()
    .reset_index(name="n_partidos")
)

# Nos quedamos solo con los casos donde ese día ese apellido/sexo tuvo 1 solo partido
actuo_ese_dia = actuo_ese_dia[actuo_ese_dia["n_partidos"] == 1].drop(columns="n_partidos")

# Recuperar el partido (ya es único)
actuo_ese_dia = actuo_ese_dia.merge(
    df_parlamentarios_css[["fecha", "apellido_key", "sexo", "partido"]].drop_duplicates(),
    on=["fecha", "apellido_key", "sexo"],
    how="left"
)


C:\Users\flala\AppData\Local\Temp\ipykernel_2880\2521932135.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_parlamentarios_css["fecha"] = pd.to_datetime(df_parlamentarios_css["fecha"])


In [129]:
actuo_ese_dia = actuo_ese_dia.rename(columns={"partido": "partido_dia"})

df_limpio = df_limpio.merge(
    actuo_ese_dia,
    left_on=["fecha", "apellido_key", "sexo_locutor"],
    right_on=["fecha", "apellido_key", "sexo"],
    how="left"
)

mask_pend = df_limpio["partido_imputado"].isna()
mask_dia = mask_pend & df_limpio["partido_dia"].notna()

df_limpio.loc[mask_dia, "partido_imputado"] = df_limpio.loc[mask_dia, "partido_dia"]
df_limpio.loc[mask_dia, "metodo_imputacion"] = "actuo_ese_dia"
df_limpio.loc[mask_dia, "score_match"] = 98


In [130]:
df_limpio.drop(columns=[c for c in ["sexo", "partido_dia"] if c in df_limpio.columns], inplace=True)


In [131]:
# si el apellido_key es ministro/ministra/secretario/secretaria/subsecretario y la n_legislatura es 47, 48 o 50, asignar partido "FRENTE AMPLIO"
mask_ministro = df_limpio["apellido_key"].str.contains(r"(ministro|ministra|secretario|secretaria|subsecretario)", case=False, na=False)
mask_legislatura = df_limpio["n_legislatura"].isin([47, 48, 50])
df_limpio.loc[mask_ministro & mask_legislatura, "partido_imputado"] = "FRENTE AMPLIO"
df_limpio.loc[mask_ministro & mask_legislatura, "metodo_imputacion"] = "ministro_legislatura"
df_limpio.loc[mask_ministro & mask_legislatura, "score_match"] = 99

C:\Users\flala\AppData\Local\Temp\ipykernel_2880\2574184840.py:2: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask_ministro = df_limpio["apellido_key"].str.contains(r"(ministro|ministra|secretario|secretaria|subsecretario)", case=False, na=False)


In [132]:
# se quitan observaciones en donde hay errores de tipeo en presidente. apellido_key es presidene/presisdenta/presdiente/prsidente
df_limpio = df_limpio[~df_limpio["apellido_key"].str.contains(r"(presidene|presisdenta|presdiente|prsidente)", case=False, na=False)]

C:\Users\flala\AppData\Local\Temp\ipykernel_2880\3104235643.py:2: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df_limpio = df_limpio[~df_limpio["apellido_key"].str.contains(r"(presidene|presisdenta|presdiente|prsidente)", case=False, na=False)]


In [133]:
# se agrega columna id de intervencion en df_limpio
df_limpio["id_intervencion"] = df_limpio.index + 1

Se imputan manualmente intervenciones problemáticas

In [134]:
# --- helper: normalizar apellido_locutor_norm para matchear robusto ---
def norm_key(s: pd.Series) -> pd.Series:
    return (
        s.fillna("")
         .astype(str)
         .str.lower()
         .str.strip()
         .str.replace("_", " ", regex=False)
         .str.replace(r"\s+", " ", regex=True)
    )

# máscara de pendientes (NO tocar los que ya tienen partido)
pend = df_limpio["partido_imputado"].isna()

# key normalizada para comparar
k = norm_key(df_limpio["apellido_locutor_norm"])

# ------------------------------------------------------------
# 1) ELIMINAR
# ------------------------------------------------------------
to_drop_keys = {
    "alfie", "alem", "angelo", "aviaga", "bert", "calloia", "capurro",
    "castro", "delgrossi", "emicuri", "fernandez", "garce", "hendler",
    "percovich", "roselli", "salom",
}

drop_mask_keys = pend & k.isin(to_drop_keys)
drop_mask_id   = df_limpio["id_intervencion"] == 11361

print("=== DROPS ===")
print(f"Por apellido (to_drop_keys):  {drop_mask_keys.sum()} intervenciones")
print(f"Por id_intervencion (11361):  {drop_mask_id.sum()} intervenciones")
print("\nDetalle por apellido:")
print(df_limpio.loc[drop_mask_keys, "apellido_locutor_norm"].value_counts().to_string())

drop_mask = drop_mask_keys | drop_mask_id
print(f"\nTotal dropeado: {drop_mask.sum()} intervenciones")

# drop por key (solo pendientes)  ← estas dos líneas son las originales
df_limpio = df_limpio.loc[~drop_mask].copy()

# Recalcular pend/k porque df_limpio cambió
pend = df_limpio["partido_imputado"].isna()
k = norm_key(df_limpio["apellido_locutor_norm"])

# ------------------------------------------------------------
# 2) ASIGNACIONES MANUALES (solo pendientes)
#    Usar etiquetas EXACTAS: FRENTE AMPLIO / NACIONAL / COLORADO / CABILDO ABIERTO
# ------------------------------------------------------------
manual_party = {
    # COLORADO
    "acosta y lara": "COLORADO",
    "coutinho": "COLORADO",
    "gurmendez": "COLORADO",
    "pintado": "COLORADO",
    "punales": "COLORADO",
    "viera": "COLORADO",

    # FRENTE AMPLIO
    "andrade": "FRENTE AMPLIO",
    "apezteguia": "FRENTE AMPLIO",
    "bergara": "FRENTE AMPLIO",
    "berti": "FRENTE AMPLIO",
    "breccia": "FRENTE AMPLIO",
    "canepa": "FRENTE AMPLIO",
    "casaravilla": "FRENTE AMPLIO",
    "coya": "FRENTE AMPLIO",
    "dalmas": "FRENTE AMPLIO",
    "de leon": "FRENTE AMPLIO",
    "garibaldi": "FRENTE AMPLIO",
    "gonzalez": "FRENTE AMPLIO",
    "herrera": "FRENTE AMPLIO",
    "kechichian": "FRENTE AMPLIO",
    "layera": "FRENTE AMPLIO",
    "lorenzo": "FRENTE AMPLIO",
    "mahia": "FRENTE AMPLIO",
    "martinez": "FRENTE AMPLIO",
    "mendez": "FRENTE AMPLIO",
    "montero": "FRENTE AMPLIO",
    "netto": "FRENTE AMPLIO",
    "pasadores": "FRENTE AMPLIO",
    "penaloza": "FRENTE AMPLIO",
    "pereyra": "FRENTE AMPLIO",
    "riet": "FRENTE AMPLIO",
    "sendic": "FRENTE AMPLIO",
    "seoane": "FRENTE AMPLIO",
    "silva": "FRENTE AMPLIO",
    "topolansky": "FRENTE AMPLIO",
    "xavier": "FRENTE AMPLIO",

    # NACIONAL
    "blas": "NACIONAL",
    "botana": "NACIONAL",
    "bustillo": "NACIONAL",
    "curbelo": "NACIONAL",
    "delgado sicco": "NACIONAL",
    "ferres": "NACIONAL",
    "fossati": "NACIONAL",
    "pena": "NACIONAL",
    "rodriguez": "NACIONAL",
    "saravia": "NACIONAL",
    "secretario": "NACIONAL",
    "abreu": "NACIONAL",          # por "senador abreu"
    "senador abreu": "NACIONAL",

    # Roles / cargos
    "subsecretario": "NACIONAL",
    "ministra de economia y finanzas": "NACIONAL",
    "subsecretario de economia y finanzas": "NACIONAL",

    "ministro de salud publica": "CABILDO ABIERTO", 
    "ministra de salud publica": "CABILDO ABIERTO", 
    "subsecretario de salud publica": "CABILDO ABIERTO",

    "ministro de ambiente": "COLORADO",
    "subsecretario de ambiente": "COLORADO",

    "ministro": "NACIONAL"
}

# aplicar asignaciones directas (excepto ministra de salud publica que va condicional)
for key, party in manual_party.items():
    if party is None:
        continue
    m = pend & (k == key)
    df_limpio.loc[m, "partido_imputado"] = party
    df_limpio.loc[m, "metodo_imputacion"] = "manual"
    df_limpio.loc[m, "score_match"] = 100


# ------------------------------------------------------------
# 3) Correcciones de sexo y de "SEÑOR/SEÑORA" SOLO si están pendientes
# ------------------------------------------------------------
def fix_locutor_prefix(mask, from_word, to_word):
    # soporta SEÑOR/SEÑORA y SENOR/SENORA por si quedaron sin ñ
    loc = df_limpio.loc[mask, "locutor"].fillna("").astype(str)

    loc = loc.str.replace(rf"^{from_word}\b", to_word, regex=True)
    # variantes sin ñ
    loc = loc.str.replace(rf"^{from_word.replace('Ñ','N')}\b", to_word.replace("Ñ","N"), regex=True)

    df_limpio.loc[mask, "locutor"] = loc

# andrade/botana/mahia: sexo M y si dice SEÑORA -> SEÑOR
for key in ["andrade", "botana", "mahia"]:
    m = (df_limpio["partido_imputado"].isna()) & (k == key)
    if m.any():
        df_limpio.loc[m, "sexo_locutor"] = "M"
        fix_locutor_prefix(m, "SEÑORA", "SEÑOR")

# kechichian/topolansky/dalmas: sexo F y si dice SEÑOR -> SEÑORA
for key in ["kechichian", "topolansky", "dalmas"]:
    m = (df_limpio["partido_imputado"].isna()) & (k == key)
    if m.any():
        df_limpio.loc[m, "sexo_locutor"] = "F"
        fix_locutor_prefix(m, "SEÑOR", "SEÑORA")


# ------------------------------------------------------------
# 4) correciones de apellido
# ------------------------------------------------------------
m_borda = df_limpio["partido_imputado"].isna() & (k == "bordabeerry")
if m_borda.any():
    df_limpio.loc[m_borda, "apellido_locutor_norm"] = "bordaberry"
    df_limpio.loc[m_borda, "apellido_key"] = "bordaberry"
    df_limpio.loc[m_borda, "partido_imputado"] = "COLORADO"
    df_limpio.loc[m_borda, "metodo_imputacion"] = "manual"
    df_limpio.loc[m_borda, "score_match"] = 100

m_couti = df_limpio["partido_imputado"].isna() & (k == "coutinhno")
if m_couti.any():
    df_limpio.loc[m_couti, "apellido_locutor_norm"] = "coutinho"
    df_limpio.loc[m_couti, "apellido_key"] = "coutinho"
    df_limpio.loc[m_couti, "partido_imputado"] = "COLORADO"
    df_limpio.loc[m_couti, "metodo_imputacion"] = "manual"
    df_limpio.loc[m_couti, "score_match"] = 100

m_carrera = df_limpio["partido_imputado"].isna() & (k == "carrrera")
if m_carrera.any():
    df_limpio.loc[m_carrera, "apellido_locutor_norm"] = "carrera"
    df_limpio.loc[m_carrera, "apellido_key"] = "carrera"
    df_limpio.loc[m_carrera, "partido_imputado"] = "FRENTE AMPLIO"
    df_limpio.loc[m_carrera, "metodo_imputacion"] = "manual"
    df_limpio.loc[m_carrera, "score_match"] = 100
    
# ------------------------------------------------------------
# 5) (Opcional) sanity check: ver qué quedó pendiente todavía
# ------------------------------------------------------------
pend_rest = df_limpio[df_limpio["partido_imputado"].isna()]
print("Pendientes restantes:", len(pend_rest))
print(pend_rest["apellido_locutor_norm"].value_counts().head(30))

=== DROPS ===
Por apellido (to_drop_keys):  32 intervenciones
Por id_intervencion (11361):  1 intervenciones

Detalle por apellido:
apellido_locutor_norm
percovich    7
emicuri      7
alfie        3
delgrossi    3
angelo       2
salom        1
roselli      1
castro       1
capurro      1
hendler      1
calloia      1
aviaga       1
alem         1
fernandez    1
garce        1

Total dropeado: 33 intervenciones
Pendientes restantes: 0
Series([], Name: count, dtype: int64)


In [97]:
pend_rest

,file_name,n_legislatura,cuerpo,fecha,locutor,encabezado,intervencion,n_palabras,n_caracteres,año,sexo_locutor,apellido_locutor_norm,apellido_key,partido_imputado,metodo_imputacion,score_match,id_intervencion


In [98]:
#se exporta df_limpio a jsonl
df_limpio.to_json("../data/processed/df_final_etiquetados.jsonl", orient="records", lines=True)
